In [9]:
import sys
from pathlib import Path

# Ajouter le répertoire src pour les imports
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import open3d as o3d

In [10]:
# Pipeline Vidéo → Scène 3D (TSDF)
from video_to_3d import run_pipeline

# Chemin vers la vidéo (modifier selon votre fichier)
VIDEO_PATH = Path(r"C:\Users\mvm\Downloads\MicrosoftTeams-video.mp4")  # ou Path("..") / "data" / "sample_video.mp4"
OUTPUT_DIR = Path(r"C:\Users\mvm\OneDrive - Group Seco\Desktop") / "output" / "tsdf_scene"

# Si la vidéo n'existe pas, créer une vidéo de test
if not VIDEO_PATH.exists():
    import cv2
    import numpy as np
    VIDEO_PATH = Path("..") / "data" / "test_video.avi"
    VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
    out = cv2.VideoWriter(str(VIDEO_PATH), cv2.VideoWriter_fourcc(*"XVID"), 10, (320, 240))
    for _ in range(50):
        out.write((np.random.rand(240, 320, 3) * 255).astype("uint8"))
    out.release()
    print("Vidéo de test créée:", VIDEO_PATH)

In [11]:
# Exécution du pipeline avec Depth Pro pour les cartes de profondeur
from video_to_3d import make_depth_pro_fn

# Créer le modèle depth (nécessite le checkpoint depth_pro.pt)
depth_fn = make_depth_pro_fn()  # ou make_depth_pro_fn(device="cpu")

mesh = run_pipeline(
    VIDEO_PATH,
    OUTPUT_DIR,
    frame_stride=5,
    max_frames=100,
    use_colmap=False,  # True si COLMAP installé
    depth_model_fn=depth_fn,
    voxel_size=0.01,
)

print("Mesh généré:", mesh)
print("Vertices:", len(mesh.vertices), "Triangles:", len(mesh.triangles))

Mesh généré: TriangleMesh with 1645996 points and 3288604 triangles.
Vertices: 1645996 Triangles: 3288604


In [12]:
# Visualisation
o3d.visualization.draw_geometries(
    [mesh],
    window_name="Scène 3D (TSDF)",
    width=1024,
    height=768,
)